# Employee Performance Analysis

#### Code: 10281
### INX Future Inc.

# FEATURE SELECTION + SMOTE

## Importing Libraries

In [43]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split,  GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel

from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier 

RandomForestClassifier → used for feature importance<br>
SelectFromModel → selects important features<br>
SMOTE → handles class imbalance

## Loading Final Dataset

In [44]:
df = pd.read_excel("C:\\Users\\navee\\Downloads\\INX_Future_Inc_Employee_Performance_CDS_Project\\Data\\Processed\\final_employee_performance_for_model.xlsx")
df

,Age,Gender,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,...,EmpJobRole_Senior Developer,EmpJobRole_Senior Manager R&D,EmpJobRole_Technical Architect,EmpJobRole_Technical Lead,MaritalStatus_Married,MaritalStatus_Single,ExperienceRatio,TotalSatisfaction,AvgYearsPerRole,PerformanceRating
0,-0.541458,0.809427,0.102061,0.103000,1.177978,-0.543569,0.379608,-0.060955,1.151824,-0.683034,...,-0.212829,-0.112509,-0.0766,-0.180838,-0.916783,1.457738,1.186706,1.556760,-0.084878,3
1,1.109888,0.809427,0.591464,1.061145,1.177978,-1.187042,0.379608,-0.060955,-1.574386,-0.264636,...,-0.212829,-0.112509,-0.0766,-0.180838,-0.916783,1.457738,-0.876605,0.546697,-0.424759,3
2,0.339260,0.809427,-0.509693,1.061145,1.177978,-0.890055,-1.035081,0.842082,-1.574386,0.990556,...,-0.212829,-0.112509,-0.0766,-0.180838,1.090771,-0.685994,1.000543,0.041665,-0.052508,4
3,0.449349,0.809427,0.102061,1.061145,-0.656641,0.347393,-1.035081,2.648157,1.151824,0.153761,...,-0.212829,-0.112509,-0.0766,-0.180838,-0.916783,-0.685994,0.989877,-0.463366,1.436496,3
4,2.541054,0.809427,0.836165,1.061145,-1.573950,0.891870,0.379608,-0.060955,-1.574386,2.245748,...,-0.212829,-0.112509,-0.0766,-0.180838,-0.916783,1.457738,-1.419582,-0.968398,-0.613582,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,-1.091906,-1.235442,-0.754394,-1.813288,1.177978,0.248397,1.794297,-0.060955,1.151824,-0.683034,...,-0.212829,-0.112509,-0.0766,-0.180838,-0.916783,-0.685994,1.000543,1.051728,-0.311465,4
1196,0.008991,0.809427,0.102061,-0.855144,1.177978,0.693878,1.794297,-0.963992,1.151824,0.153761,...,4.698609,-0.112509,-0.0766,-0.180838,-0.916783,1.457738,-1.354424,0.546697,-0.311465,3
1197,1.440157,0.809427,2.304373,-1.813288,1.177978,0.396891,1.794297,-0.963992,0.243087,-0.683034,...,4.698609,-0.112509,-0.0766,-0.180838,1.090771,-0.685994,1.341842,1.051728,0.796296,3
1198,-0.321278,-1.235442,-0.020290,0.103000,1.177978,-0.989050,-1.035081,0.842082,-0.665650,-0.683034,...,-0.212829,-0.112509,-0.0766,-0.180838,-0.916783,1.457738,0.795763,0.546697,-0.311465,3


## Splitting Features & Target

In [45]:
X = df.drop('PerformanceRating', axis=1)
y = df['PerformanceRating']

## Train-Test Split

In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)

stratify=y → maintains class distribution<br>
Important for imbalanced datasets

## FEATURE SELECTION

In [47]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

selector = SelectFromModel(rf, threshold='median', prefit=True)

# FIX: Convert back to DataFrame to preserve structure
X_train_selected = pd.DataFrame(
    selector.transform(X_train),
    columns=X_train.columns[selector.get_support()]
)

X_test_selected = pd.DataFrame(
    selector.transform(X_test),
    columns=X_train_selected.columns
)

C:\Users\navee\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\navee\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


## Applying SMOTE

In [48]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_selected, y_train
)

# TOP 4 MODELS (Best Choice) :

- Logistic Regression (fast baseline)<br>
- Random Forest (strong + stable)<br>
- Decision Tree (interpretable)<br>
- Gradient Boosting (high performance)

# Model Training 

## K-Fold Setup (Optimized)

In [49]:
kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# MODEL 1: Logistic Regression

In [50]:
lr = LogisticRegression(max_iter=500)

lr_params = {
    'C': [0.1, 1],
    'solver': ['lbfgs']
}

lr_grid = GridSearchCV(
    lr, lr_params,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

,estimator,LogisticRegre...(max_iter=500)
,param_grid,"{'C': [0.1, 1], 'solver': ['lbfgs']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


# MODEL 2: Decision Tree

In [51]:
dt = DecisionTreeClassifier(random_state=42)

dt_params = {
    'max_depth': [5, 10],
    'min_samples_split': [2, 5]
}

dt_grid = GridSearchCV(
    dt, dt_params,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'max_depth': [5, 10], 'min_samples_split': [2, 5]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


# MODEL 3: Random Forest

In [52]:
rf = RandomForestClassifier(random_state=42)

rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10]
}

rf_grid = GridSearchCV(
    rf, rf_params,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [5, 10], 'n_estimators': [50, 100]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,50


# MODEL 4: Gradient Boosting

In [53]:
gb = GradientBoostingClassifier(random_state=42)

gb_params = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1]
}

gb_grid = GridSearchCV(
    gb, gb_params,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.05, 0.1], 'n_estimators': [50, 100]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


# Collecting Training Results

In [54]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting'],
    'Best Score (CV)': [
        lr_grid.best_score_,
        dt_grid.best_score_,
        rf_grid.best_score_,
        gb_grid.best_score_
    ],
    'Best Params': [
        str(lr_grid.best_params_),
        str(dt_grid.best_params_),
        str(rf_grid.best_params_),
        str(gb_grid.best_params_)
    ]
})

print(results)

                 Model  Best Score (CV)  \
0  Logistic Regression         0.809375   
1        Decision Tree         0.908333   
2        Random Forest         0.881250   
3    Gradient Boosting         0.934375   

                                   Best Params  
0                {'C': 0.1, 'solver': 'lbfgs'}  
1     {'max_depth': 5, 'min_samples_split': 2}  
2        {'max_depth': 10, 'n_estimators': 50}  
3  {'learning_rate': 0.05, 'n_estimators': 50}  


## SAVING TEST DATA

In [55]:
X_test.to_excel("X_test.xlsx", index=False)
y_test.to_excel("y_test.xlsx", index=False)

Saves test features and labels separately<br>
Used later for prediction

## SAVING TRAINED MODELS

## Import Libraries

In [56]:
import joblib

## Saving All Models

In [57]:
joblib.dump(lr_grid.best_estimator_, "logistic_model.pkl")
joblib.dump(dt_grid.best_estimator_, "decision_tree_model.pkl")
joblib.dump(rf_grid.best_estimator_, "random_forest_model.pkl")
joblib.dump(gb_grid.best_estimator_, "gradient_boosting_model.pkl")

['gradient_boosting_model.pkl']

.pkl files store trained models<br>
best_estimator_ → saves optimized model